# 4 · Tables

A table is one value, a `polars.DataFrame`. To work through it one record at a time, put a Split after it. Everything after the Split runs once per record, until a node that takes a `Series` or a `Table` gathers the results.

In [ ]:
from typing import TypedDict

import polars as pl
from conductor import Graph, GraphRecord, NodeRecord, NodeRegistry, Series, Table, node
from conductor_nodes.series import split
from conductor_nodes.tables import collect


class Document(TypedDict):
    name: str


@node("documents", title="Documents", description="Lists the documents to read.")
def documents() -> Table[Document]:
    return pl.DataFrame({"name": ["a.pdf", "b.pdf"]})


@node("lines", title="Lines", description="Reads the lines of a document.")
def lines(name: str) -> Series[str]:
    return ["first line", "second line"] if name == "a.pdf" else []


@node("label", title="Label", description="Writes a line in capital letters after the name of its document.")
def label(line: str, document: str) -> str:
    return f"{document}: {line.upper()}"


@node("count", title="Count", description="Counts the lines.")
def count(values: Series[str]) -> int:
    return len(values)


registry = NodeRegistry()
registry.add(documents, lines, label, count, split, collect)

## Split, then once per item

`series-split` and `table-collect` come from `conductor-nodes`. `series-split` takes the table apart: its result is a `Series` of records, and `series-split.result.name` is the `name` field of each.

- `lines` takes one name, so it runs once per document. It returns a `Series` too, so `label` runs once per line of each document.
- `label` also reads the document's name. That value is made outside the lines, so it is simply there on every line.
- `how_many` takes a `Series[str]`, so it gathers: it runs once per document and gets every line of that document.
- `overview` gathers the records into one table again, with a new column, `lines`, for the count made beside each.

In [ ]:
graph = Graph.compile(GraphRecord(
    nodes=[
        NodeRecord(id="documents", node="documents"),
        NodeRecord(id="series-split", node="series-split"),
        NodeRecord(id="lines", node="lines"),
        NodeRecord(id="label", node="label"),
        NodeRecord(id="how_many", node="count"),
        NodeRecord(id="overview", node="table-collect", values={"columns": ["lines"]}),
    ],
    edges=[
        ("documents.result", "series-split.value"),
        ("series-split.result.name", "lines.name"),
        ("lines.result", "label.line"),
        ("series-split.result.name", "label.document"),
        ("lines.result", "how_many.values"),
        ("series-split.result", "overview.records"),
        ("how_many.result", "overview.lines"),
    ],
), registry)
print(graph.problems)
for node_id, runs in graph.runs_on.items():
    print(f"  {node_id:<10} {runs.msg}")

## What comes out

`run.results` has every call of a node, in order. `b.pdf` has no lines, so `label` is never called for it, and `how_many` is called with an empty list and counts 0. The table has a row per document, `b.pdf` included.

In [ ]:
results = graph.run().results
print(results["label"]["result"])
print(results["how_many"]["result"])
print(results["overview"]["result"])

## Forgetting the Split

A table never runs anything per record by itself. Wire a table, or one of its columns, straight into an input of one value, and loading says what to do:

In [ ]:
forgot = Graph.compile(GraphRecord(
    nodes=[NodeRecord(id="documents", node="documents"), NodeRecord(id="lines", node="lines")],
    edges=[("documents.result.name", "lines.name")],
), registry)
for problem in forgot.problems:
    print(problem.type, problem.msg)

## Blanks

A column that may be blank says so: `email: str | None`. A blank reaches a node only if its input says `| None` too. Otherwise the node isn't called for that item, its output is blank there, and `run.not_called` says which item and input.

In [ ]:
class Contact(TypedDict):
    email: str | None


@node("contacts", title="Contacts", description="Lists the contacts.")
def contacts() -> Table[Contact]:
    return pl.DataFrame({"email": ["ida@a.dk", None]}, schema={"email": pl.String})


@node("domain", title="Domain", description="Picks the domain out of an email address.")
def domain(email: str) -> str:
    return email.split("@")[1]


@node("has-email", title="Has email", description="Tells whether a contact has an email address.")
def has_email(email: str | None) -> bool:
    return email is not None


registry.add(contacts, domain, has_email)
blanks = Graph.compile(GraphRecord(
    nodes=[
        NodeRecord(id="contacts", node="contacts"),
        NodeRecord(id="series-split", node="series-split"),
        NodeRecord(id="domain", node="domain"),
        NodeRecord(id="has_email", node="has-email"),
    ],
    edges=[("contacts.result", "series-split.value"), ("series-split.result.email", "domain.email"), ("series-split.result.email", "has_email.email")],
), registry)
run = blanks.run()
print(run.results["domain"]["result"], run.results["has_email"]["result"])
print(run.not_called)